# Lab 19 — One-click OpenAI + Neo4j Aura + Golden 50

Mục tiêu: **fresh Colab runtime → Runtime > Run all → nhận ZIP cuối**, không cần cell test/resume thủ công.

## Colab Secrets bắt buộc
- `OPENAI_API_KEY`
- `HF_TOKEN`
- `NEO4J_CREDENTIALS` (**khuyến nghị**): paste nguyên nội dung file credentials Aura đã download vào một secret này.

Nếu không dùng `NEO4J_CREDENTIALS`, runner vẫn hỗ trợ bộ secret rời: `NEO4J_URI`, `NEO4J_USERNAME` hoặc `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`. Khi `NEO4J_CREDENTIALS` tồn tại, runner ưu tiên block này để tránh username/password bị lệch giữa các secret cũ.

Optional: `LLM_MODEL` / `JUDGE_MODEL` (default `gpt-4.1-mini`). Nếu AuraDB chỉ dành riêng cho lab và an toàn để xoá graph cũ, đặt `LAB_RESET_GRAPH=1`.

Runner preflight Neo4j + OpenAI trước khi coreference/extraction. Nếu Aura trả `Unauthorized`, đó là credential mismatch và cell dừng ngay trước khi tốn API extraction.

In [ ]:
#@title 1 — Clone latest main (safe to rerun)
%cd /content
!rm -rf /content/lab19
!git clone -q https://github.com/QuocKhanhLuong/K3-Track3-Lab19-GraphRAG-2A202601713-LuongQuocKhanh.git /content/lab19
%cd /content/lab19
!git log -1 --oneline


In [ ]:
#@title 2 — Load reference notebook definitions + first 5,000 source rows
import json
from pathlib import Path

reference_path = Path('/content/lab19/Day19_GraphRAG_vs_FlatRAG_Production_Lab_Guide.ipynb')
nb = json.loads(reference_path.read_text(encoding='utf-8'))

for idx, cell in enumerate(nb['cells']):
    if cell.get('cell_type') != 'code':
        continue
    src = ''.join(cell.get('source', []))

    # Official Golden 50 is based on the first 5,000 source rows.
    src = src.replace('LIMIT_ROWS = 1_000_000', 'LIMIT_ROWS = 5000')
    src = src.replace('LIMIT_MB = 300', 'LIMIT_MB = 80')
    src = src.replace('PRIORITIZE_MB = True', 'PRIORITIZE_MB = False')

    result = get_ipython().run_cell(src)
    if getattr(result, 'error_before_exec', None):
        raise result.error_before_exec
    if getattr(result, 'error_in_exec', None):
        raise result.error_in_exec

print('Reference notebook loaded. Dataset:', DATA_PATH)


In [ ]:
#@title 3 — ONE CLICK: preflight → full solution → official Golden 50 → ZIP
from pathlib import Path

def exec_file(path):
    path = Path(path)
    code = compile(path.read_text(encoding='utf-8'), str(path), 'exec')
    exec(code, globals(), globals())

# `exec` is deliberate: any exception stops this cell immediately.
# No stale partial state is allowed to continue into solution/Golden.
exec_file('/content/lab19/openai_runtime_patch.py')
preflight_services()

exec_file('/content/lab19/colab_solution.py')

# Reached only if the full solution completed successfully.
exec_file('/content/lab19/official_golden_eval.py')


## Done
Nếu Cell 3 hoàn tất, Colab tự tải `/content/lab19_submission_official50.zip`. Hai CSV rubric chính trong `outputs/` là kết quả official Golden 50.